# GOAPified Adaptive RAG

This notebook demonstrates how **LangGoap** replaces the hardcoded conditional edges in
[LangGraph's Adaptive RAG](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_adaptive_rag/)
with formal GOAP planning.

## Original vs GOAPified

| Aspect | Original LangGraph | GOAPified (LangGoap) |
|--------|-------------------|---------------------|
| Routing | Hardcoded `route_question`, `decide_to_generate`, `grade_generation` edges | A\* planner discovers optimal action sequence |
| Retrieval strategy | LLM decides vectorstore vs web search | Planner picks lowest-cost path automatically |
| Failure recovery | Manual conditional edge back to transform_query | Observer detects deviation, triggers replanning |
| Plan verification | None — LLM may hallucinate routing | A\* guarantees plan is achievable from preconditions |

## Architecture

Each RAG component becomes a **GOAP action** with typed preconditions and effects:

```
retrieve_documents:  has_question → has_documents      (cost=1)
web_search:          has_question → has_documents      (cost=2)
grade_documents:     has_documents → has_relevant_documents
generate_answer:     has_relevant_documents → answer_ready
transform_query:     has_question → query_transformed
```

The A\* planner finds the cheapest path from the initial state to the goal.

In [1]:
from typing import Any

from langgoap import ActionSpec, GoalSpec, GoapGraph, ReplanStrategy

## Define RAG Actions

Each action simulates a RAG component. In production, these would call real
vector stores, web search APIs, and LLMs.

In [2]:
def retrieve_documents(ws: dict[str, Any]) -> dict[str, Any]:
    """Retrieve documents from a vector store."""
    question = ws.get("question", "")
    docs = [
        {"page_content": f"Relevant info about {question}", "source": "vectorstore"},
        {"page_content": "Background context", "source": "vectorstore"},
    ]
    return {"has_documents": True, "documents": docs}


def web_search(ws: dict[str, Any]) -> dict[str, Any]:
    """Search the web for information."""
    question = ws.get("question", "")
    docs = [
        {"page_content": f"Web result for: {question}", "source": "web"},
    ]
    return {"has_documents": True, "documents": docs}


def grade_documents(ws: dict[str, Any]) -> dict[str, Any]:
    """Grade retrieved documents for relevance."""
    docs = ws.get("documents", [])
    return {"has_relevant_documents": True, "relevant_documents": docs}


def generate_answer(ws: dict[str, Any]) -> dict[str, Any]:
    """Generate an answer from relevant documents."""
    docs = ws.get("relevant_documents", ws.get("documents", []))
    content = "; ".join(d.get("page_content", "") for d in docs)
    return {
        "answer_ready": True,
        "generation": f"Generated answer from {len(docs)} docs: {content}",
    }


def transform_query(ws: dict[str, Any]) -> dict[str, Any]:
    """Rewrite question for better retrieval."""
    question = ws.get("question", "")
    return {
        "has_question": True,
        "question": f"[rewritten] {question}",
        "query_transformed": True,
    }

## Build GOAP Action Specifications

Each action is wrapped in an `ActionSpec` with:
- **preconditions**: what must be true in world state before the action can execute
- **effects**: what the action guarantees after execution (used by A\* planner)
- **cost**: numeric cost used by A\* to find the optimal plan

In [3]:
actions = [
    ActionSpec(
        name="retrieve_documents",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        cost=1.0,  # Cheaper: prefer vectorstore
        execute=retrieve_documents,
    ),
    ActionSpec(
        name="web_search",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        cost=2.0,  # More expensive: fallback
        execute=web_search,
    ),
    ActionSpec(
        name="grade_documents",
        preconditions={"has_documents": True},
        effects={"has_relevant_documents": True},
        cost=1.0,
        execute=grade_documents,
    ),
    ActionSpec(
        name="generate_answer",
        preconditions={"has_relevant_documents": True},
        effects={"answer_ready": True},
        cost=1.0,
        execute=generate_answer,
    ),
    ActionSpec(
        name="transform_query",
        preconditions={"has_question": True},
        effects={"query_transformed": True},
        cost=1.5,
        execute=transform_query,
    ),
]

## Happy Path: Vectorstore Retrieval

The planner discovers the optimal path: `retrieve_documents → grade_documents → generate_answer`.
It prefers vectorstore retrieval (cost=1) over web search (cost=2).

In [4]:
result = GoapGraph(actions=actions).invoke(
    goal=GoalSpec(conditions={"answer_ready": True}),
    world_state={
        "has_question": True,
        "question": "What are agent memory types?",
    },
)

print(f"Status: {result['status']}")
print(f"Answer: {result['world_state']['generation']}")
print(f"Actions executed: {[h.action_name for h in result['execution_history'] if h.success]}")

Status: goal_achieved
Answer: Generated answer from 2 docs: Relevant info about What are agent memory types?; Background context
Actions executed: ['retrieve_documents', 'grade_documents', 'generate_answer']


## Cost-Based Routing

When both retrieval paths are available, A\* picks the lower-cost option.
This replaces the original's LLM-based `route_question` conditional edge.

In [5]:
# Exaggerate cost difference to demonstrate routing
cost_demo_actions = [
    ActionSpec(
        name="retrieve_documents",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        cost=1.0,
        execute=retrieve_documents,
    ),
    ActionSpec(
        name="web_search",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        cost=5.0,  # Much more expensive
        execute=web_search,
    ),
    ActionSpec(
        name="grade_documents",
        preconditions={"has_documents": True},
        effects={"has_relevant_documents": True},
        execute=grade_documents,
    ),
    ActionSpec(
        name="generate_answer",
        preconditions={"has_relevant_documents": True},
        effects={"answer_ready": True},
        execute=generate_answer,
    ),
]

result = GoapGraph(actions=cost_demo_actions).invoke(
    goal=GoalSpec(conditions={"answer_ready": True}),
    world_state={"has_question": True, "question": "agent architectures"},
)

successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Planner chose: {successful[0]}")
print(f"Full sequence: {successful}")

Planner chose: retrieve_documents
Full sequence: ['retrieve_documents', 'grade_documents', 'generate_answer']


## Web Search Fallback

When the vectorstore is unavailable (action not registered), the planner
automatically routes through web search — no conditional edges needed.

In [6]:
web_only_actions = [
    ActionSpec(
        name="web_search",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        execute=web_search,
    ),
    ActionSpec(
        name="grade_documents",
        preconditions={"has_documents": True},
        effects={"has_relevant_documents": True},
        execute=grade_documents,
    ),
    ActionSpec(
        name="generate_answer",
        preconditions={"has_relevant_documents": True},
        effects={"answer_ready": True},
        execute=generate_answer,
    ),
]

result = GoapGraph(actions=web_only_actions).invoke(
    goal=GoalSpec(conditions={"answer_ready": True}),
    world_state={"has_question": True, "question": "2024 NFL draft picks"},
)

print(f"Status: {result['status']}")
print(f"Actions: {[h.action_name for h in result['execution_history'] if h.success]}")
print(f"Source: {result['world_state']['generation']}")

Status: goal_achieved
Actions: ['web_search', 'grade_documents', 'generate_answer']
Source: Generated answer from 1 docs: Web result for: 2024 NFL draft picks


## Replanning on Failure

When an action raises an exception, the observer detects the failure and
triggers replanning. This replaces the original's `decide_to_generate`
and `grade_generation` conditional edges.

In [7]:
call_count = {"retrieve": 0}


def flaky_retrieve(ws: dict[str, Any]) -> dict[str, Any]:
    """Fails on first call, succeeds on subsequent calls."""
    call_count["retrieve"] += 1
    if call_count["retrieve"] == 1:
        raise ConnectionError("Vector store unavailable")
    return retrieve_documents(ws)


replan_actions = [
    ActionSpec(
        name="retrieve_documents",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        cost=1.0,
        execute=flaky_retrieve,
    ),
    ActionSpec(
        name="web_search",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        cost=2.0,
        execute=web_search,
    ),
    ActionSpec(
        name="grade_documents",
        preconditions={"has_documents": True},
        effects={"has_relevant_documents": True},
        execute=grade_documents,
    ),
    ActionSpec(
        name="generate_answer",
        preconditions={"has_relevant_documents": True},
        effects={"answer_ready": True},
        execute=generate_answer,
    ),
]

result = GoapGraph(actions=replan_actions).invoke(
    goal=GoalSpec(conditions={"answer_ready": True}),
    world_state={"has_question": True, "question": "agent memory"},
)

print(f"Status: {result['status']}")
print(f"Replan count: {result['replan_count']}")
failures = [h for h in result["execution_history"] if not h.success]
print(f"Failures: {[(f.action_name, f.error) for f in failures]}")
print(f"Final answer: {result['world_state']['generation'][:80]}...")

Action 'retrieve_documents' failed: Vector store unavailable


Status: goal_achieved
Replan count: 1
Failures: [('retrieve_documents', 'Vector store unavailable')]
Final answer: Generated answer from 2 docs: Relevant info about agent memory; Background conte...


## Two-Tier State: Planning Flags + Rich Data

LangGoap separates **planning state** (hashable boolean flags for A\*) from
**execution context** (rich data like document lists, user metadata).
Both flow through the pipeline, but only planning flags are used by A\*.

In [8]:
result = GoapGraph(actions=actions).invoke(
    goal=GoalSpec(conditions={"answer_ready": True}),
    world_state={
        "has_question": True,
        "question": "What is GOAP planning?",
        "user_id": "test-user-123",  # Non-planning metadata
    },
)

ws = result["world_state"]
print(f"Documents (list): {type(ws['documents']).__name__}, count={len(ws['documents'])}")
print(f"Relevant docs (list): {type(ws['relevant_documents']).__name__}")
print(f"User metadata preserved: user_id={ws['user_id']}")

Documents (list): list, count=2
Relevant docs (list): list
User metadata preserved: user_id=test-user-123


## Unreachable Goals

When no action chain can reach the goal, the planner reports `no_plan`
instead of hallucinating a path.

In [9]:
# Only retrieve_documents is available — can't reach answer_ready
incomplete_actions = [
    ActionSpec(
        name="retrieve_documents",
        preconditions={"has_question": True},
        effects={"has_documents": True},
        execute=retrieve_documents,
    ),
]

result = GoapGraph(actions=incomplete_actions).invoke(
    goal=GoalSpec(conditions={"answer_ready": True}),
    world_state={"has_question": True},
)

print(f"Status: {result['status']}")

A* found no plan for goal {'answer_ready': True}


Status: no_plan
